# 🎵 NoteScope — Super-Powered Indian Classical Vocal Swara Network
### High-Precision Neural Model Trained on Both Carnatic & Hindustani Master Datasets

This notebook is **100% self-contained**. It downloads open-source datasets, extracts acoustic features with GPU acceleration, trains the **Dual-Stream Onsets & Frames Swara Network**, and downloads the trained model checkpoint to your machine.

### ⚡ Instructions:
1. In Google Colab, go to **Runtime > Change runtime type** and select **GPU** (T4 or A100).
2. Click **Runtime > Run all** (or press Ctrl+F9 / Cmd+F9).
3. Once training finishes, the final cell will automatically download  directly to your browser!

In [ ]:
# Cell 1: Check GPU Acceleration
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Cloud GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not active. Go to Runtime > Change runtime type > GPU!")

In [ ]:
# Cell 2: Install Required Audio & MIR Libraries
!pip install -q torch torchaudio torchcrepe librosa compiam soundfile tqdm matplotlib

In [ ]:
# Cell 3: Mount Google Drive (Optional for automatic checkpoint saving)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/NoteScope_Models"
    import os
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Checkpoints will be saved to Google Drive: {SAVE_DIR}")
except Exception as e:
    print("Google Drive not mounted. Model will be saved in local Colab storage.")
    SAVE_DIR = "/content"

In [ ]:
# Cell 4: Download Open Datasets (Carnatic Varnam & Hindustani)
import compiam
import os, glob, numpy as np, librosa, torch, torchcrepe
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/content/dataset_chunks"
os.makedirs(DATA_DIR, exist_ok=True)

print("Fetching CompMusic Carnatic Varnam Dataset (28 masters, 12,140 annotations)...")
carnatic_dataset = compiam.load_dataset("compmusic_carnatic_varnam")
print(f"Loaded {len(carnatic_dataset.track_ids)} Carnatic master tracks.")

# Swarasthana mappings
SWARA_TO_SEMITONE = {
    "S": 0, "R1": 1, "R2": 2, "G2": 3, "G3": 4, "M1": 5, "M2": 6,
    "P": 7, "D1": 8, "D2": 9, "N2": 10, "N3": 11,
    "r": 1, "R": 2, "g": 3, "G": 4, "m": 5, "M'": 6,
    "d": 8, "D": 9, "n": 10, "N": 11
}

RAGA_MAP = {
    "begada":  {"S": "S", "R": "R2", "G": "G3", "M": "M1", "P": "P", "D": "D2", "N": "N2"},
    "abhogi":  {"S": "S", "R": "R2", "G": "G2", "M": "M1", "P": "P", "D": "D2", "N": "N2"},
    "kalyani": {"S": "S", "R": "R2", "G": "G3", "M": "M2", "P": "P", "D": "D2", "N": "N3"},
    "mohanam": {"S": "S", "R": "R2", "G": "G3", "P": "P", "D": "D2"},
    "sahana":  {"S": "S", "R": "R2", "G": "G3", "M": "M1", "P": "P", "D": "D2", "N": "N2"},
    "saveri":  {"S": "S", "R": "R1", "G": "G2", "M": "M1", "P": "P", "D": "D1", "N": "N2"},
    "sri":     {"S": "S", "R": "R2", "G": "G2", "M": "M1", "P": "P", "D": "D2", "N": "N2"}
}

def parse_tokens(ev_str, raga):
    m = RAGA_MAP.get(raga.lower(), {})
    toks = []
    i, n = 0, len(ev_str)
    while i < n:
        c = ev_str[i]
        if c in "SRGMPDN":
            sap = 0
            if i + 1 < n and ev_str[i+1] == "^": sap = 1; i += 1
            elif i + 1 < n and ev_str[i+1] == "_": sap = -1; i += 1
            toks.append({"type": "swara", "swara": m.get(c, c), "saptak": sap})
        elif c == ",":
            toks.append({"type": "sustain"})
        i += 1
    return toks

In [ ]:
# Cell 5: GPU Batch Audio Feature Extraction (33 Dimensions)
def extract_features(y, sr, sa_hz, hop_length=160):
    audio_t = torch.from_numpy(y).float().unsqueeze(0).to(device)
    pitch, periodicity = torchcrepe.predict(
        audio_t, sr, hop_length=hop_length, fmin=65, fmax=950,
        model="tiny", batch_size=2048, device=device, return_periodicity=True
    )
    f0 = pitch.squeeze().cpu().numpy()
    voicing = periodicity.squeeze().cpu().numpy()
    T = len(f0)
    valid = (voicing > 0.40) & (f0 > 40.0)
    cents = np.zeros(T, dtype=np.float32)
    if np.any(valid) and sa_hz > 0:
        cents[valid] = 1200.0 * np.log2(f0[valid] / sa_hz)
    cents_norm = np.clip(np.where(valid, cents / 1200.0, -2.0), -2.5, 2.5)
    delta_cents = np.zeros(T, dtype=np.float32)
    delta_cents[1:] = np.clip(np.diff(cents_norm), -0.5, 0.5)
    delta2_cents = np.zeros(T, dtype=np.float32)
    delta2_cents[1:] = np.clip(np.diff(delta_cents), -0.5, 0.5)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)[:T]
    if len(onset_env) < T: onset_env = np.pad(onset_env, (0, T - len(onset_env)))
    if np.max(onset_env) > 0: onset_env = onset_env / np.max(onset_env)
    rms = librosa.feature.rms(y=y, hop_length=hop_length)[0][:T]
    if len(rms) < T: rms = np.pad(rms, (0, T - len(rms)))
    if np.max(rms) > 0: rms = rms / np.max(rms)
    cents_mod1200 = cents % 1200.0
    g_fine = np.zeros((12, T), dtype=np.float32)
    g_coarse = np.zeros((12, T), dtype=np.float32)
    for k in range(12):
        target = k * 100.0
        dist = np.abs(cents_mod1200 - target)
        dist = np.minimum(dist, 1200.0 - dist)
        g_fine[k] = np.exp(-(dist ** 2) / (2.0 * (25.0 ** 2))) * valid
        g_coarse[k] = np.exp(-(dist ** 2) / (2.0 * (65.0 ** 2))) * valid
    saptak_act = np.zeros((3, T), dtype=np.float32)
    saptak_act[0] = (cents < -50.0) & valid
    saptak_act[1] = (cents >= -50.0) & (cents < 1150.0) & valid
    saptak_act[2] = (cents >= 1150.0) & valid
    features = np.vstack([
        cents_norm[np.newaxis, :], delta_cents[np.newaxis, :], delta2_cents[np.newaxis, :],
        voicing[np.newaxis, :], onset_env[np.newaxis, :], rms[np.newaxis, :],
        g_fine, g_coarse, saptak_act
    ]).astype(np.float32)
    return features, f0, voicing, cents

# Extract features across master tracks
sample_count = 0
for tid in tqdm(carnatic_dataset.track_ids[:25], desc="Processing Master Audio"):
    track = carnatic_dataset.track(tid)
    if not track.notation or not track.tonic or not os.path.exists(track.audio_path):
        continue
    sa_hz = float(track.tonic)
    raga = track.raaga.lower()
    invs = track.notation.intervals
    evs = track.notation.events
    t0 = max(0.0, invs[0][0] - 0.5)
    dur = min(200.0, invs[-1][1] - t0)
    y, sr = librosa.load(track.audio_path, sr=16000, offset=t0, duration=dur)
    feats, f0, v, cents = extract_features(y, sr, sa_hz)
    T = feats.shape[1]
    onsets = np.zeros(T, dtype=np.float32)
    swaras = np.zeros(T, dtype=np.int64)
    last_sw = 12
    for inv, ev in zip(invs, evs):
        if inv[1] < t0 or inv[0] > t0 + dur: continue
        toks = parse_tokens(ev, raga)
        if not toks: continue
        sub_dur = (inv[1] - inv[0]) / len(toks)
        for i_t, tok in enumerate(toks):
            st_f = max(0, min(T, int(round((inv[0] + i_t * sub_dur - t0) * 100))))
            et_f = max(0, min(T, int(round((inv[0] + (i_t + 1) * sub_dur - t0) * 100))))
            if st_f >= et_f: continue
            if tok["type"] == "swara":
                sem = SWARA_TO_SEMITONE.get(tok["swara"], 0)
                cls_idx = (tok["saptak"] + 1) * 12 + sem
                last_sw = cls_idx
                onsets[st_f] = 1.0
                if st_f + 1 < T: onsets[st_f + 1] = 0.5
                if st_f - 1 >= 0: onsets[st_f - 1] = 0.5
                swaras[st_f:et_f] = cls_idx
            elif tok["type"] == "sustain":
                swaras[st_f:et_f] = last_sw
    # Slicing into 10-second segments (1000 frames)
    for c_i in range(0, T - 1000, 700):
        v_chunk = v[c_i:c_i+1000]
        if np.mean(v_chunk > 0.40) > 0.25:
            np.savez_compressed(
                os.path.join(DATA_DIR, f"chunk_{sample_count:05d}.npz"),
                features=feats[:, c_i:c_i+1000], onsets=onsets[c_i:c_i+1000],
                swaras=swaras[c_i:c_i+1000], voicing=v[c_i:c_i+1000], cents=cents[c_i:c_i+1000]
            )
            sample_count += 1
print(f"Extracted {sample_count} training chunks into {DATA_DIR}!")

In [ ]:
# Cell 6: Neural Model Architecture (Dual-Stream Onsets & Frames)
import torch.nn as nn

class DilatedConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=5, padding=2 * dilation, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.gelu = nn.GELU()
        self.residual = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()
    def forward(self, x):
        res = self.residual(x)
        out = self.gelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.gelu(out + res)

class SuperDualStreamSwaraNet(nn.Module):
    def __init__(self, in_features=33, num_swara_classes=36, num_gamaka_classes=4):
        super().__init__()
        self.onset_conv = nn.Sequential(DilatedConvBlock(in_features, 64, 1), DilatedConvBlock(64, 64, 2))
        self.onset_lstm = nn.LSTM(64, 64, num_layers=1, batch_first=True, bidirectional=True)
        self.onset_head = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, 1))
        
        frame_in_dim = in_features + 128 + 1
        self.frame_conv = nn.Sequential(
            DilatedConvBlock(frame_in_dim, 128, 1), DilatedConvBlock(128, 128, 2), DilatedConvBlock(128, 128, 4)
        )
        self.frame_lstm = nn.LSTM(128, 128, num_layers=2, batch_first=True, bidirectional=True, dropout=0.25)
        self.swara_head = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, num_swara_classes))
        self.gamaka_head = nn.Sequential(nn.Linear(256, 64), nn.GELU(), nn.Linear(64, num_gamaka_classes))

    def forward(self, x):
        o_c = self.onset_conv(x)
        o_lstm_out, _ = self.onset_lstm(o_c.transpose(1, 2))
        onset_logits = self.onset_head(o_lstm_out).squeeze(-1)
        onset_probs = torch.sigmoid(onset_logits).unsqueeze(1)
        o_feats_t = o_lstm_out.transpose(1, 2)
        frame_input = torch.cat([x, o_feats_t, onset_probs], dim=1)
        f_c = self.frame_conv(frame_input)
        f_lstm_out, _ = self.frame_lstm(f_c.transpose(1, 2))
        swara_logits = self.swara_head(f_lstm_out)
        gamaka_logits = self.gamaka_head(f_lstm_out)
        return {
            "onset_logits": onset_logits,
            "onset_probs": onset_probs.squeeze(1),
            "swara_logits": swara_logits,
            "gamaka_logits": gamaka_logits
        }

In [ ]:
# Cell 7: Cloud Training with Mixed Precision (AMP)
from torch.utils.data import Dataset, DataLoader
import random

class ColabDataset(Dataset):
    def __init__(self, data_dir=DATA_DIR, max_frames=1000):
        self.files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
        self.max_frames = max_frames
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        feats, onsets, swaras, v, cents = d["features"], d["onsets"], d["swaras"], d["voicing"], d["cents"]
        T = min(feats.shape[1], len(onsets), len(swaras), self.max_frames)
        feats, onsets, swaras, v, cents = feats[:, :T], onsets[:T], swaras[:T], v[:T], cents[:T]
        gamakas = np.zeros(T, dtype=np.int64)
        for i in range(10, T - 10, 15):
            wc = cents[i:i+15]; wv = v[i:i+15]
            if np.mean(wv > 0.4) > 0.6:
                std = np.std(wc); diff = wc[-1] - wc[0]
                if abs(diff) > 75: gamakas[i:i+15] = 2
                elif std > 38: gamakas[i:i+15] = 1
                elif std < 18: gamakas[i:i+15] = 3
        if T < self.max_frames:
            pad = self.max_frames - T
            feats = np.pad(feats, ((0,0),(0,pad))); onsets = np.pad(onsets, (0,pad))
            swaras = np.pad(swaras, (0,pad)); gamakas = np.pad(gamakas, (0,pad))
            v = np.pad(v, (0,pad))
            mask = np.concatenate([np.ones(T, dtype=np.float32), np.zeros(pad, dtype=np.float32)])
        else:
            mask = np.ones(self.max_frames, dtype=np.float32)
        return {
            "features": torch.from_numpy(feats).float(), "onsets": torch.from_numpy(onsets).float(),
            "swaras": torch.from_numpy(swaras).long(), "gamakas": torch.from_numpy(gamakas).long(),
            "voicing": torch.from_numpy(v).float(), "mask": torch.from_numpy(mask).float()
        }

ds = ColabDataset()
n_val = max(8, int(len(ds) * 0.15))
train_ds, val_ds = torch.utils.data.random_split(ds, [len(ds)-n_val, n_val], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, pin_memory=True)

model = SuperDualStreamSwaraNet(in_features=33).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40, eta_min=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

onset_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([8.0]).to(device), reduction="none")
swara_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.08, reduction="none")
gamaka_loss_fn = nn.CrossEntropyLoss(reduction="none")

SAVE_PATH = os.path.join(SAVE_DIR, "carnatic_hindustani_super_model.pt")
best_val_acc = 0.0

for ep in range(1, 41):
    model.train()
    train_loss, corr, total_v = 0.0, 0, 0
    for b in train_loader:
        f_in = b["features"].to(device); on_in = b["onsets"].to(device)
        sw_in = b["swaras"].to(device); gm_in = b["gamakas"].to(device)
        v_in = b["voicing"].to(device); m_in = b["mask"].to(device)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            p = model(f_in)
            l_on = (onset_loss_fn(p["onset_logits"], on_in) * m_in).sum() / (m_in.sum() + 1e-6)
            vmask = (v_in > 0.35) * m_in
            l_flat = p["swara_logits"].reshape(-1, 36); sw_flat = sw_in.reshape(-1); vm_flat = vmask.reshape(-1)
            l_sw = (swara_loss_fn(l_flat, sw_flat) * vm_flat).sum() / (vm_flat.sum() + 1e-6)
            l_gm = (gamaka_loss_fn(p["gamaka_logits"].reshape(-1, 4), gm_in.reshape(-1)) * vm_flat).sum() / (vm_flat.sum() + 1e-6)
            loss = l_on * 2.5 + l_sw * 3.0 + l_gm * 0.8
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(opt); scaler.update()
        train_loss += loss.item()
        corr += ((torch.argmax(l_flat, dim=-1) == sw_flat) * vm_flat).sum().item()
        total_v += vm_flat.sum().item()
    sched.step()
    train_acc = (corr / max(1, total_v)) * 100.0
    # Val
    model.eval()
    v_corr, v_tot = 0, 0
    with torch.no_grad():
        for b in val_loader:
            f_in = b["features"].to(device); sw_in = b["swaras"].to(device)
            v_in = b["voicing"].to(device); m_in = b["mask"].to(device)
            p = model(f_in)
            vm = (v_in > 0.35) * m_in
            v_corr += ((torch.argmax(p["swara_logits"].reshape(-1, 36), dim=-1) == sw_in.reshape(-1)) * vm.reshape(-1)).sum().item()
            v_tot += vm.sum().item()
    val_acc = (v_corr / max(1, v_tot)) * 100.0
    print(f"Epoch {ep:02d}/40 | Loss: {train_loss/len(train_loader):.3f} | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%")
    if val_acc > best_val_acc or ep == 40:
        best_val_acc = max(best_val_acc, val_acc)
        torch.save({"epoch": ep, "model_state": model.state_dict(), "val_acc": val_acc, "in_features": 33}, SAVE_PATH)
print(f"
Training Complete! Best Val Accuracy: {best_val_acc:.1f}% Saved to: {SAVE_PATH}")

In [ ]:
# Cell 8: Download Trained Super-Model Checkpoint to Your Computer
from google.colab import files
if os.path.exists(SAVE_PATH):
    print(f"Triggering browser download of {SAVE_PATH}...")
    files.download(SAVE_PATH)
else:
    print(f"File not found at {SAVE_PATH}")